# Stage-3 mock wedges: 3D cosmic web (two new sky boxes)

Interactive **Plotly** 3D scatter of Abacus SecondGen **stage-3** mock galaxies in two new survey wedges, colored by **T-Web** environment class (void / wall / filament / cluster).

| Wedge | NPZ | Manifest bounds |
|-------|-----|-----------------|
| **Wedge 1** | `staged_mock_wedge_stage3_ra120_140_dec16p5_26p7_z0p2_0p3_rs7.npz` | RA 120-140, Dec 16.5-26.7, z 0.2-0.3 |
| **Wedge 2** | `staged_mock_wedge_stage3_wedge2_ra128_138_dec18_25_z0_0p5_rs7.npz` | RA 128-138, Dec 18-25, z 0.0-0.5 |

**Classification**: NPZ `cls`, or `(lambda1,2,3 > lambda_thr).sum()` with **`lambda_thr = 0.2`**.

Positions: **RA, DEC, Z** to comoving **Mpc** via **Planck18** (not Mpc/h).

> **Wedge 2:** recommended box is **RA 127-133, Dec 20-24**; built NPZ uses **128-138 / 18-25**. No `ra127_133` NPZ on disk — see `CACHE_TRAINING_NEW_WEDGES.md`.

> **Graphs:** not built yet for these tags (graph cell below).


In [ ]:
from pathlib import Path
import json
import numpy as np

WEDGE_DIR = Path("/pscratch/sd/d/dkololgi/abacus/SecondGen_Mocks/ph000/wedge").resolve()
GRAPH_DIR = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions").resolve()

WEDGES = [
    (
        "Wedge 1 (stage3, z 0.2-0.3)",
        WEDGE_DIR / "staged_mock_wedge_stage3_ra120_140_dec16p5_26p7_z0p2_0p3_rs7.npz",
        WEDGE_DIR / "staged_mock_wedge_stage3_ra120_140_dec16p5_26p7_z0p2_0p3_rs7.manifest.json",
        "staged_mock_wedge_stage3_ra120_140_dec16p5_26p7_z0p2_0p3_rs7",
    ),
    (
        "Wedge 2 (stage3, z 0-0.5)",
        WEDGE_DIR / "staged_mock_wedge_stage3_wedge2_ra128_138_dec18_25_z0_0p5_rs7.npz",
        WEDGE_DIR / "staged_mock_wedge_stage3_wedge2_ra128_138_dec18_25_z0_0p5_rs7.manifest.json",
        "staged_mock_wedge_stage3_wedge2_ra128_138_dec18_25_z0_0p5_rs7",
    ),
]

WEDGE2_REC_NPZ = WEDGE_DIR / "staged_mock_wedge_stage3_wedge2_ra127_133_dec20_24_z0_0p5_rs7.npz"
LAMBDA_THRESHOLD = 0.2
MAX_POINTS = 250_000
SEED = 0
MPC_H = False
NOTEBOOK_DIR = Path("/global/homes/d/dkololgi/TNG/Illustris/workflows/abacus_tweb").resolve()
OUT_HTML = NOTEBOOK_DIR / "visualize_staged_mock_stage3_new_wedges_3d.html"

missing = [p for _, p, _, _ in WEDGES if not p.exists()]
if missing:
    raise FileNotFoundError("Missing NPZ(s): " + ", ".join(map(str, missing)))

print("Wedge dir:", WEDGE_DIR)
print("lambda_thr:", LAMBDA_THRESHOLD, "MAX_POINTS:", MAX_POINTS)
print(
    "ra127_133 wedge2 exists:",
    WEDGE2_REC_NPZ.exists(),
    WEDGE2_REC_NPZ.name if WEDGE2_REC_NPZ.exists() else "using " + WEDGES[1][1].name,
)

for label, npz_path, manifest_path, _ in WEDGES:
    man = json.loads(manifest_path.read_text(encoding="utf-8"))
    b = man.get("bounds") or man.get("wedge", {})
    print(
        f"{label}: n_gal={man.get('n_gal')} "
        f"RA [{b.get('ra_min')},{b.get('ra_max')}] "
        f"Dec [{b.get('dec_min')},{b.get('dec_max')}] "
        f"z [{b.get('z_min')},{b.get('z_max')}]"
    )
    print("  class_fractions:", man.get("class_fractions"))


In [ ]:
CLASS_NAMES = np.array(["void", "wall", "filament", "cluster"])
COLOR_MAP = {0: "#4C78A8", 1: "#F58518", 2: "#54A24B", 3: "#E45756"}

def _pick_array(data, candidates):
    for key in candidates:
        if key in data:
            return np.asarray(data[key])
    return None

def classify_from_lambdas(lam, threshold):
    return (np.asarray(lam, dtype=np.float64) > float(threshold)).sum(axis=1).astype(np.int8)

def load_staged_npz(path, lambda_threshold=LAMBDA_THRESHOLD):
    data = np.load(path)
    ra = np.asarray(data["ra"], dtype=np.float64)
    dec = np.asarray(data["dec"], dtype=np.float64)
    zz = np.asarray(data["z"], dtype=np.float64)
    cls = _pick_array(data, ("cls", "CLS", "cweb_class", "CWEB_CLASS"))
    if cls is None:
        l1 = _pick_array(data, ("lambda1", "LAMBDA1"))
        l2 = _pick_array(data, ("lambda2", "LAMBDA2"))
        l3 = _pick_array(data, ("lambda3", "LAMBDA3"))
        cls = classify_from_lambdas(np.stack([l1, l2, l3], axis=1), lambda_threshold)
    else:
        cls = np.asarray(cls, dtype=np.int8)
    fr = np.bincount(cls.astype(np.int64), minlength=4).astype(np.float64)
    fr /= max(1.0, float(cls.size))
    return {
        "path": Path(path),
        "ra": ra,
        "dec": dec,
        "z": zz,
        "cls": cls,
        "fractions": fr,
        "n": int(ra.size),
    }

def sky_to_xyz(ra_deg, dec_deg, z, mpc_h=MPC_H):
    from astropy.cosmology import Planck18 as cosmo

    ra_rad = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec_rad = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    dist = cosmo.comoving_distance(np.asarray(z, dtype=np.float64)).value
    if mpc_h:
        dist *= float(cosmo.h)
    x = dist * np.cos(dec_rad) * np.cos(ra_rad)
    y = dist * np.cos(dec_rad) * np.sin(ra_rad)
    z3 = dist * np.sin(dec_rad)
    return x, y, z3

def subsample_indices(n, max_points, seed):
    idx = np.arange(n, dtype=np.int64)
    if max_points and n > int(max_points):
        idx = np.sort(
            np.random.default_rng(int(seed)).choice(idx, int(max_points), replace=False)
        )
    return idx

def fraction_annotation(fr, title_prefix=""):
    body = "<br>".join(f"{CLASS_NAMES[k]}: {fr[k]*100:.2f}%" for k in range(4))
    return f"{title_prefix}{body}" if title_prefix else body


In [ ]:
loaded = []
for label, npz_path, manifest_path, _ in WEDGES:
    rec = load_staged_npz(npz_path)
    rec["label"] = label
    rec["manifest"] = json.loads(manifest_path.read_text(encoding="utf-8"))
    loaded.append(rec)
    print(f"{label}: N={rec['n']:,} fractions={rec['fractions']}")


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

units = "Mpc/h" if MPC_H else "Mpc"

def make_wedge_scatter3d(rec, seed_offset=0, show=True):
    ra, dec, zz, cls = rec["ra"], rec["dec"], rec["z"], rec["cls"]
    idx = subsample_indices(ra.size, MAX_POINTS, SEED + seed_offset)
    x, y, z3 = sky_to_xyz(ra[idx], dec[idx], zz[idx])
    colors = [COLOR_MAP[int(c)] for c in cls[idx]]
    b = rec["manifest"].get("bounds") or rec["manifest"].get("wedge", {})
    title = (
        f"{rec['label']}<br><sup>{rec['path'].name} | N={rec['n']:,} | "
        f"Nplot={idx.size:,} | RA [{b.get('ra_min')},{b.get('ra_max')}] "
        f"Dec [{b.get('dec_min')},{b.get('dec_max')}] z [{b.get('z_min')},{b.get('z_max')}] "
        f"| lambda_thr={LAMBDA_THRESHOLD:g}</sup>"
    )
    fig = go.Figure(
        go.Scatter3d(
            x=x,
            y=y,
            z=z3,
            mode="markers",
            marker=dict(size=2, opacity=0.6, color=colors),
            hoverinfo="skip",
        )
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=f"x [{units}]",
            yaxis_title=f"y [{units}]",
            zaxis_title=f"z [{units}]",
            aspectmode="data",
        ),
        template="plotly_white",
        height=750,
    )
    fig.add_annotation(
        text="Full wedge<br>" + fraction_annotation(rec["fractions"]),
        xref="paper",
        yref="paper",
        x=0.99,
        y=0.99,
        showarrow=False,
        align="right",
        bgcolor="rgba(255,255,255,0.75)",
    )
    if show:
        fig.show()
    return fig

for k, rec in enumerate(loaded):
    make_wedge_scatter3d(rec, seed_offset=k, show=True)


In [ ]:
fig_grid = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[r["label"] for r in loaded],
    horizontal_spacing=0.04,
)
for k, rec in enumerate(loaded):
    idx = subsample_indices(rec["ra"].size, MAX_POINTS, SEED + 50 + k)
    x, y, z3 = sky_to_xyz(rec["ra"][idx], rec["dec"][idx], rec["z"][idx])
    colors = [COLOR_MAP[int(c)] for c in rec["cls"][idx]]
    fig_grid.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z3,
            mode="markers",
            marker=dict(size=1.5, opacity=0.55, color=colors),
            showlegend=False,
        ),
        row=1,
        col=k + 1,
    )
    fig_grid.add_annotation(
        text=f"N={rec['n']:,}<br>" + fraction_annotation(rec["fractions"]),
        xref="paper",
        yref="paper",
        x=0.02 + 0.5 * k,
        y=0.98,
        showarrow=False,
        align="left",
        font=dict(size=9),
        bgcolor="rgba(255,255,255,0.75)",
    )
fig_grid.update_layout(
    title=f"Stage-3 mock wedges (lambda_thr={LAMBDA_THRESHOLD:g}, {units})",
    template="plotly_white",
    height=700,
)
for sn in ("scene", "scene2"):
    if sn in fig_grid.layout:
        fig_grid.layout[sn].update(
            xaxis_title=f"x [{units}]",
            yaxis_title=f"y [{units}]",
            zaxis_title=f"z [{units}]",
            aspectmode="data",
        )
fig_grid.write_html(str(OUT_HTML), include_plotlyjs="cdn")
print("Wrote", OUT_HTML)
fig_grid.show()


In [ ]:
import plotly.express as px
import pandas as pd

rows = [
    {"wedge": rec["label"], "class": str(CLASS_NAMES[k]), "fraction": float(rec["fractions"][k])}
    for rec in loaded
    for k in range(4)
]
fig_bar = px.bar(
    pd.DataFrame(rows),
    x="wedge",
    y="fraction",
    color="class",
    barmode="group",
    color_discrete_map={str(n): COLOR_MAP[i] for i, n in enumerate(CLASS_NAMES)},
    title=f"Class fractions (lambda_thr={LAMBDA_THRESHOLD:g})",
)
fig_bar.update_layout(template="plotly_white", height=450, yaxis_tickformat=".1%")
fig_bar.show()


## Graph overlay (Delaunay edges)

Checks for `*_edges_combined_idx.npy` and `*_cugraph_gnn_arrays.npz` under `graph_constructions/`.

In [ ]:
MAX_EDGES_PLOT = 50_000

def graph_artifacts(prefix):
    edges = GRAPH_DIR / f"{prefix}_edges_combined_idx.npy"
    gnn = GRAPH_DIR / f"{prefix}_cugraph_gnn_arrays.npz"
    meta = GRAPH_DIR / f"{prefix}_cugraph_gnn_metadata.json"
    xyz = GRAPH_DIR / f"{prefix}_points_xyz.npy"
    return {
        "prefix": prefix,
        "edges": edges if edges.is_file() else None,
        "gnn": gnn if gnn.is_file() else None,
        "meta": meta if meta.is_file() else None,
        "xyz": xyz if xyz.is_file() else None,
        "has_graph": edges.is_file() and gnn.is_file(),
    }

graph_status = [dict(graph_artifacts(p), label=l) for l, _, _, p in WEDGES]
for st in graph_status:
    print(st["label"], "has_graph=", st["has_graph"], "edges=", st["edges"])


In [ ]:
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

if not any(st["has_graph"] for st in graph_status):
    display(
        Markdown(
            "**Graph not built yet** for either new wedge.\n\n"
            f"Expected under `{GRAPH_DIR}/<prefix>_edges_combined_idx.npy` and "
            f"`<prefix>_cugraph_gnn_arrays.npz`.\n\n"
            "See `CACHE_TRAINING_NEW_WEDGES.md` in ph000/wedge/."
        )
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, rec in zip(axes, loaded):
        idx = subsample_indices(rec["ra"].size, min(MAX_POINTS, 80_000), SEED + 200)
        ax.scatter(
            rec["ra"][idx],
            rec["dec"][idx],
            c=rec["cls"][idx],
            s=0.3,
            cmap="tab10",
            vmin=0,
            vmax=3,
            alpha=0.5,
        )
        b = rec["manifest"].get("bounds", {})
        ax.set_xlim(b.get("ra_min"), b.get("ra_max"))
        ax.set_ylim(b.get("dec_min"), b.get("dec_max"))
        ax.set_xlabel("RA [deg]")
        ax.set_ylabel("Dec [deg]")
        ax.set_title(rec["label"] + " (2D placeholder)")
        ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    plt.show()
    for st in graph_status:
        tpl = GRAPH_DIR / f"{st['prefix']}_edges_combined_idx.npy"
        print(f"Future: edges = np.load('{tpl}')  # (E,2) local indices, same order as NPZ")
else:
    for st, rec in zip(graph_status, loaded):
        if not st["has_graph"]:
            continue
        edges = np.load(st["edges"])
        xyz_all = (
            np.load(st["xyz"])
            if st["xyz"]
            else np.stack(sky_to_xyz(rec["ra"], rec["dec"], rec["z"]), axis=1)
        )
        n = xyz_all.shape[0]
        node_idx = subsample_indices(n, MAX_POINTS, SEED + 300)
        node_set = set(node_idx.tolist())
        mask = np.array([(int(u) in node_set and int(v) in node_set) for u, v in edges])
        e_sub = edges[mask]
        if e_sub.shape[0] > MAX_EDGES_PLOT:
            e_sub = e_sub[
                np.random.default_rng(SEED + 301).choice(
                    e_sub.shape[0], MAX_EDGES_PLOT, replace=False
                )
            ]
        xe, ye, ze = [], [], []
        for u, v in e_sub:
            u, v = int(u), int(v)
            xe += [xyz_all[u, 0], xyz_all[v, 0], None]
            ye += [xyz_all[u, 1], xyz_all[v, 1], None]
            ze += [xyz_all[u, 2], xyz_all[v, 2], None]
        fig_g = go.Figure()
        fig_g.add_trace(
            go.Scatter3d(
                x=xe,
                y=ye,
                z=ze,
                mode="lines",
                line=dict(width=1, color="rgba(120,120,120,0.25)"),
                hoverinfo="skip",
            )
        )
        colors = [COLOR_MAP[int(rec["cls"][i])] for i in node_idx]
        fig_g.add_trace(
            go.Scatter3d(
                x=xyz_all[node_idx, 0],
                y=xyz_all[node_idx, 1],
                z=xyz_all[node_idx, 2],
                mode="markers",
                marker=dict(size=2, color=colors, opacity=0.7),
            )
        )
        fig_g.update_layout(
            title=st["label"] + " (nodes + subsampled edges)",
            scene=dict(aspectmode="data"),
            template="plotly_white",
            height=750,
        )
        fig_g.show()
